In [3]:
import numpy as np
import pandas as pd
import ccxt
import pandas_ta as ta
from sklearn.model_selection import train_test_split


### 1. Get the Datasets

In [4]:
def get_data(symbol: str, timeframe, limit=1000):
    ex = ccxt.binance()
    ohlcv = ex.fetch_ohlcv(symbol, timeframe, limit=limit)
    df = pd.DataFrame(ohlcv, columns=['timestamp', 'Open', 'High', 'Low', 'Close', 'Volume'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
    return df


In [5]:
df = get_data('ETHUSDT', '1m')
df


,timestamp,Open,High,Low,Close,Volume
0,2025-11-17 00:46:00+00:00,3118.15,3124.85,3118.15,3123.81,322.3428
1,2025-11-17 00:47:00+00:00,3123.81,3125.33,3122.19,3123.32,233.1965
2,2025-11-17 00:48:00+00:00,3123.31,3129.67,3123.31,3128.48,380.2743
3,2025-11-17 00:49:00+00:00,3128.48,3130.26,3124.67,3128.17,343.2960
4,2025-11-17 00:50:00+00:00,3128.16,3133.77,3125.69,3133.21,177.3765
...,...,...,...,...,...,...
995,2025-11-17 17:21:00+00:00,3112.19,3112.47,3105.44,3109.51,359.9330
996,2025-11-17 17:22:00+00:00,3109.51,3117.50,3108.86,3113.95,293.9583
997,2025-11-17 17:23:00+00:00,3113.95,3117.15,3108.65,3114.93,446.5687
998,2025-11-17 17:24:00+00:00,3114.93,3116.00,3110.91,3110.91,368.2663


In [6]:
df['CCI'] = ta.cci(df['High'], df['Low'], df['Close'], length=14)
df['CMO'] = ta.cmo(df['Close'], length=14)


In [7]:
df['Target'] = np.where(df['Close'].shift(-1) > df['Close'], 1, 0)


In [8]:
df = df.dropna()
df.tail(20)


,timestamp,Open,High,Low,Close,Volume,CCI,CMO,Target
980,2025-11-17 17:06:00+00:00,3129.01,3129.01,3122.51,3125.25,656.6899,-176.284711,8.438656,0
981,2025-11-17 17:07:00+00:00,3125.25,3128.56,3121.34,3123.66,398.7108,-158.283166,4.773894,1
982,2025-11-17 17:08:00+00:00,3123.65,3129.53,3123.07,3129.53,283.6434,-78.818777,16.053408,1
983,2025-11-17 17:09:00+00:00,3129.53,3133.00,3128.84,3130.40,531.2694,-2.610414,17.611056,0
984,2025-11-17 17:10:00+00:00,3130.40,3132.38,3123.08,3123.08,266.7728,-95.212659,0.683278,0
985,2025-11-17 17:11:00+00:00,3123.08,3123.83,3116.04,3117.52,611.3145,-209.951567,-9.921931,0
986,2025-11-17 17:12:00+00:00,3117.52,3117.53,3113.50,3116.20,548.9817,-208.193554,-12.284169,0
987,2025-11-17 17:13:00+00:00,3116.20,3117.50,3113.80,3113.81,312.8232,-169.231969,-16.551271,1
988,2025-11-17 17:14:00+00:00,3113.80,3116.99,3113.23,3115.37,168.8633,-129.906382,-12.697545,0
989,2025-11-17 17:15:00+00:00,3115.37,3115.37,3105.22,3106.90,1030.6318,-155.375578,-26.841535,1


### 3. Model Building
#### 3.1. Defining X and y

In [9]:
# Get a list of columns to keep
keep_columns = ['High', 'Low', 'Open', 'Volume', 'Close', 'CCI', 'CMO']
X = df[keep_columns]
y = df['Target']


In [10]:
X


,High,Low,Open,Volume,Close,CCI,CMO
14,3138.07,3134.98,3136.33,162.4230,3136.63,80.942163,46.550472
15,3138.38,3134.38,3136.62,198.2273,3136.57,72.026047,46.207435
16,3142.01,3136.58,3136.58,261.3172,3138.21,123.402879,49.674930
17,3140.36,3131.39,3138.20,172.6890,3132.20,-8.027431,19.320392
18,3135.29,3128.54,3132.19,359.6528,3132.12,-104.091443,18.974510
...,...,...,...,...,...,...,...
995,3112.47,3105.44,3112.19,359.9330,3109.51,-85.779480,-14.010318
996,3117.50,3108.86,3109.51,293.9583,3113.95,-27.065861,-5.349093
997,3117.15,3108.65,3113.95,446.5687,3114.93,-9.320905,-3.480471
998,3116.00,3110.91,3114.93,368.2663,3110.91,-12.874110,-10.493862


In [11]:
y


14     0
15     1
16     0
17     0
18     0
      ..
995    1
996    1
997    0
998    0
999    0
Name: Target, Length: 986, dtype: int32

#### 3.2. Split the Dataset

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#### 3.3. Model Building

In [13]:
from sklearn.ensemble import RandomForestClassifier
fr = RandomForestClassifier()
fr.fit(X_train, y_train)


RandomForestClassifier()

In [14]:
fr.score(X_test, y_test)


0.4444444444444444

#### 3.5. Tuning Hyperparameters

In [15]:
from sklearn.model_selection import GridSearchCV


In [16]:
grid_search = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [3, 6, 9, 12],
    'criterion': ['gini', 'entropy', 'log_loss'],
}


In [17]:
grid = GridSearchCV(RandomForestClassifier(),
                    param_grid=grid_search, cv=5, scoring='r2')


In [18]:
model_grid = grid.fit(X_train, y_train)


In [19]:
## Let's found the parameters for this model
print(f'Best hyperparameters are {model_grid.best_params_}, score = {model_grid.best_score_}')


Best hyperparameters are {'criterion': 'entropy', 'max_depth': 3, 'n_estimators': 300}, score = -0.7543121085110662


In [31]:
fr = RandomForestClassifier(criterion='entropy', max_depth=3, n_estimators=300)
fr.fit(X_train, y_train)


RandomForestClassifier(criterion='entropy', max_depth=3, n_estimators=300)

In [32]:
fr.score(X_test, y_test)


0.4797979797979798

#### 3.4. Predictions

In [33]:
pred = fr.predict(X_test)
pred


array([0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0,
       0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1,
       0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0,
       1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0,
       1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1,
       1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1,
       0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0])

In [34]:
y_test


627    1
465    0
745    0
450    0
289    0
      ..
223    0
520    0
63     1
731    0
978    1
Name: Target, Length: 198, dtype: int32

In [35]:
import pickle as pk


In [36]:
pk.dump(fr, open('crypto_model', 'wb'))


In [37]:
model = pk.load(open('crypto_model', 'rb'))
type(model)


sklearn.ensemble._forest.RandomForestClassifier